Импорты

In [1]:
import os, sys
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np

In [2]:
from src.data.split import make_time_split
from src.models.baseline import naive_forecast, seasonal_naive_forecast, moving_average_forecast
from src.models.evaluate import backtest
from src.features.lags import make_lagged_supervised
from src.models.train import train_ridge_model
from src.models.forecast import recursive_forecast

df = pd.read_csv("../data/raw/oil_sales_synth.csv")
series = df["sales"].to_numpy(dtype=float)

split = make_time_split(len(series), train_frac=0.7, val_frac=0.15)
H = 30
lags = 60

split

TimeSplit(train_end=766, val_end=930, test_end=1095)

Baseline и разделение 

In [3]:
baseline_val = {
    "naive": backtest(series, lambda h, H: naive_forecast(h, H), split.train_end, split.val_end, H, stride=7),
    "seasonal_naive_7": backtest(series, lambda h, H: seasonal_naive_forecast(h, H, season=7), split.train_end, split.val_end, H, stride=7),
    "moving_avg_7": backtest(series, lambda h, H: moving_average_forecast(h, H, window=7), split.train_end, split.val_end, H, stride=7),
}
baseline_test = {
    "naive": backtest(series, lambda h, H: naive_forecast(h, H), split.val_end, split.test_end, H, stride=7),
    "seasonal_naive_7": backtest(series, lambda h, H: seasonal_naive_forecast(h, H, season=7), split.val_end, split.test_end, H, stride=7),
    "moving_avg_7": backtest(series, lambda h, H: moving_average_forecast(h, H, window=7), split.val_end, split.test_end, H, stride=7),
}
baseline_val, baseline_test

({'naive': Metrics(mae=7.964288626754419, rmse=9.743272246613182),
  'seasonal_naive_7': Metrics(mae=4.028637277371823, rmse=5.250130860545673),
  'moving_avg_7': Metrics(mae=4.461338403466334, rmse=5.68388796191074)},
 {'naive': Metrics(mae=6.994501820030567, rmse=8.385500381889118),
  'seasonal_naive_7': Metrics(mae=3.629165465440801, rmse=4.615119209628429),
  'moving_avg_7': Metrics(mae=4.410092112975194, rmse=5.424000818474247)})

Финальная модель + backtest

In [4]:
X, y = make_lagged_supervised(series, lags=lags)
train_end_sup = split.train_end - lags

model = train_ridge_model(X[:train_end_sup], y[:train_end_sup]).model

model_val = backtest(series, lambda h, H: recursive_forecast(model, np.asarray(h, dtype=float), H, lags), split.train_end, split.val_end, H, stride=7)
model_test = backtest(series, lambda h, H: recursive_forecast(model, np.asarray(h, dtype=float), H, lags), split.val_end, split.test_end, H, stride=7)

model_val, model_test

(Metrics(mae=2.9784419878070296, rmse=4.11282051834857),
 Metrics(mae=2.611436281802127, rmse=3.418925280201136))

Таблица

In [5]:
import pandas as pd

rows = []
for name, m in baseline_val.items():
    rows.append({"model": name, "split": "val", "MAE": m.mae, "RMSE": m.rmse})
for name, m in baseline_test.items():
    rows.append({"model": name, "split": "test", "MAE": m.mae, "RMSE": m.rmse})

rows.append({"model": "ridge_lags60_recursive", "split": "val", "MAE": model_val.mae, "RMSE": model_val.rmse})
rows.append({"model": "ridge_lags60_recursive", "split": "test", "MAE": model_test.mae, "RMSE": model_test.rmse})

pd.DataFrame(rows).sort_values(["split", "MAE"])

,model,split,MAE,RMSE
7,ridge_lags60_recursive,test,2.611436,3.418925
4,seasonal_naive_7,test,3.629165,4.615119
5,moving_avg_7,test,4.410092,5.424001
3,naive,test,6.994502,8.385500
6,ridge_lags60_recursive,val,2.978442,4.112821
1,seasonal_naive_7,val,4.028637,5.250131
2,moving_avg_7,val,4.461338,5.683888
0,naive,val,7.964289,9.743272
